In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch


In [3]:
words = ['hello', 'world', 'i', 'am', 'gerald',
          'and', 'gerald', 'is', 'using', 'torch',
          'to', 'build', 'a', 'character', 'model',
          'also', 'giovanna', 'is', 'my', 'daughter',
          'and', 'she', 'is', 'very', 'cute', 'also',
          'my', 'wife', 'is', 'a', 'loving', 'person',
          'she','takes', 'care', 'of', 'me', 'and', 'our',
          'family','we', 'enjoy', 'spending', 'time', 'together',
          'for', 'some', 'reasons', 'i', 'need', 'to', 'query', 'this']

In [17]:
import string
chars = string.ascii_lowercase
stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {i: ch for ch, i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [38]:
block_size = 3
X, Y = [], []

for w in words[:3]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X, dtype=torch.int64)
Y = torch.tensor(Y, dtype=torch.int64)

hello
... -> h
..h -> e
.he -> l
hel -> l
ell -> o
llo -> .
world
... -> w
..w -> o
.wo -> r
wor -> l
orl -> d
rld -> .
i
... -> i
..i -> .


In [39]:
import torch.nn.functional as F
import torch.nn as nn

In [40]:
C = torch.randn((27, 2))

In [41]:
C[5], C[5].dtype

(tensor([-1.1928, -1.0445]), torch.float32)

In [42]:
# 1 , 27 x 27, 2 = 1, 2
F.one_hot(torch.tensor(5), num_classes=27).float() @ C # We are typecasting to float to match the dtype of C

tensor([-1.1928, -1.0445])

In [43]:
X.shape, Y.shape

(torch.Size([14, 3]), torch.Size([14]))

In [51]:
C[X]

tensor([[[ 0.1742,  0.4627],
         [ 0.1742,  0.4627],
         [ 0.1742,  0.4627]],

        [[ 0.1742,  0.4627],
         [ 0.1742,  0.4627],
         [ 0.8077,  2.1266]],

        [[ 0.1742,  0.4627],
         [ 0.8077,  2.1266],
         [-1.1928, -1.0445]],

        [[ 0.8077,  2.1266],
         [-1.1928, -1.0445],
         [ 0.6633, -0.6248]],

        [[-1.1928, -1.0445],
         [ 0.6633, -0.6248],
         [ 0.6633, -0.6248]],

        [[ 0.6633, -0.6248],
         [ 0.6633, -0.6248],
         [-0.8550, -0.9292]],

        [[ 0.1742,  0.4627],
         [ 0.1742,  0.4627],
         [ 0.1742,  0.4627]],

        [[ 0.1742,  0.4627],
         [ 0.1742,  0.4627],
         [ 1.5990, -1.1287]],

        [[ 0.1742,  0.4627],
         [ 1.5990, -1.1287],
         [-0.8550, -0.9292]],

        [[ 1.5990, -1.1287],
         [-0.8550, -0.9292],
         [-0.0515, -0.3945]],

        [[-0.8550, -0.9292],
         [-0.0515, -0.3945],
         [ 0.6633, -0.6248]],

        [[-0.0515, -0

In [49]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  8],
        [ 0,  8,  5],
        [ 8,  5, 12],
        [ 5, 12, 12],
        [12, 12, 15],
        [ 0,  0,  0],
        [ 0,  0, 23],
        [ 0, 23, 15],
        [23, 15, 18],
        [15, 18, 12],
        [18, 12,  4],
        [ 0,  0,  0],
        [ 0,  0,  9]])

In [50]:
C[0], C[0], C[0], C[0], C[0], C[8]

(tensor([0.1742, 0.4627]),
 tensor([0.1742, 0.4627]),
 tensor([0.1742, 0.4627]),
 tensor([0.1742, 0.4627]),
 tensor([0.1742, 0.4627]),
 tensor([0.8077, 2.1266]))

In [52]:
C[X].shape

torch.Size([14, 3, 2])

In [53]:
X[13, 2]

tensor(9)

In [54]:
C[X][13, 2]

tensor([-0.5323, -1.6896])

In [55]:
C[9]

tensor([-0.5323, -1.6896])

In [57]:
emb = C[X]  # (N, block_size, D)

In [58]:
w1 = torch.randn(size=(6, 100))
b1 = torch.randn(size=(100,))

In [61]:
# We can't do this emb @ w1 + b1 because the dimensions don't match
# We need to reshape emb to (N, block_size * D) to match w1
emb_reshaped = emb.view(emb.shape[0], -1)  # (N, block_size * D)
(emb_reshaped @ w1 + b1).shape  # (N, 100)

torch.Size([14, 100])

In [63]:
h = torch.tanh(emb_reshaped @ w1 + b1)  # (N, 100)
h.shape

torch.Size([14, 100])

In [64]:
W2 = torch.randn(size=(100, 27))
b2 = torch.randn(size=(27,))

In [65]:
logits = h @ W2 + b2  # (N, 27)

In [66]:
logits.shape

torch.Size([14, 27])

In [67]:
counts = logits.exp()

In [69]:
probs = counts / counts.sum(dim=1, keepdim=True)  # (N, 27)

In [71]:
probs[0].sum()

tensor(1.0000)

In [79]:
Y

tensor([ 8,  5, 12, 12, 15,  0, 23, 15, 18, 12,  4,  0,  9,  0])

In [73]:
probs.shape

torch.Size([14, 27])

In [76]:
probs[[0,1], Y[0]]  # Probability of the first character in the first word

tensor([5.9435e-06, 8.9138e-06])

In [82]:
probs[torch.arange(probs.shape[0]), Y] # probs[0].shape is 14 as with Y.shape = 14

tensor([5.9435e-06, 2.3136e-10, 7.3429e-08, 2.4671e-02, 2.3379e-13, 2.1089e-11,
        8.4012e-06, 2.0450e-14, 9.7910e-01, 9.3912e-09, 1.3024e-12, 9.0670e-01,
        6.8123e-09, 2.3816e-13])

In [83]:
loss = -probs[torch.arange(probs.shape[0]), Y].log().mean()
loss

tensor(17.5045)

In [ ]:
g = torch.generators.Generator().manual_seed(42)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn(size=(6))